# Day 44 — Retrieval evaluation: 10 queries, manual grades

Status: COMPLETE. 10 grounded queries × top-3, graded by reading the chunks:
**8/10 HIT**. Both misses diagnosed (not hidden). Grades in
`models/retrieval_eval.json`; raw hits in `models/retrieval_raw.json`.

In [1]:
import json
from pathlib import Path

ev = json.loads(Path("../models/retrieval_eval.json").read_text())
print(f"score: {ev['score']}")
print("HITs (exact-doc top-1): DVT ultrasound 0.765, carotid 0.72, ASD film 0.779,")
print("  COPD HPI 0.686, BRCA-2 0.766, plantar 0.723, migraine 0.696, disc 0.70")
print("MISSes: stress-test impression (indications crowd out results),")
print("  mitral/LVH echo (aortic-stenosis neighbors dominate)")
print("fix direction (Day 46): k=5 + result-oriented phrasing; no code change yet —")
print("  generation must never be wired to unmeasured retrieval.")

score: 8/10 top-3 hits
HITs (exact-doc top-1): DVT ultrasound 0.765, carotid 0.72, ASD film 0.779,
  COPD HPI 0.686, BRCA-2 0.766, plantar 0.723, migraine 0.696, disc 0.70
MISSes: stress-test impression (indications crowd out results),
  mitral/LVH echo (aortic-stenosis neighbors dominate)
fix direction (Day 46): k=5 + result-oriented phrasing; no code change yet —
  generation must never be wired to unmeasured retrieval.


## Findings

1. **8/10 with top-1 exact-doc hits on most** — the chunker + embeddings +
   index stack works; failures are query-side (indication-vs-result wording),
   not index-side.
2. **Both misses share a shape**: the corpus contains many *indication*
   chunks ('rule out X') that embed near *result* queries. Retrieval can't tell
   'asked about' from 'concluded' — generation grounding (Day 45) must therefore
   quote, and the eval set (Day 46) must include such near-miss traps.
3. **Measured before generating** — had we wired the LLM first, these two
   failures would have surfaced as fluent hallucinations instead of clean
   retrieval misses. Debug order matters.